# MBL simulations: Imbrie & random-field XXZ

**One notebook, one reproducible disorder realization, two solvers.**

Run the configuration and preparation cells first. Remote submission and retrieval are **opt-in**. You can reopen an existing run directory later to retrieve, analyze and replot it.

**Requirements:** `pip install -e ".[notebook]"` in the repository root. Qaptiva/QLMaaS packages and an accessible server are needed only for remote cells. See `docs/User_Guide.pdf`.

## 1. Imports and configuration

Edit this cell. The Imbrie model uses **Pauli** couplings; the XXZ model uses **spin-1/2** couplings.

Adaptive integration: `error_control=True`, **no `nsteps`**; fixed-step: set `error_control=False` **and** a positive integer `nsteps`.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from qaptiva_mbl import (RunConfig, prepare_run, load_run, submit_run,
                         retrieve_run, run_ed, compare_run, plot_run)

cfg = RunConfig(
    model="imbrie",             # "imbrie" or "xxz"
    L=8,
    boundary="pbc",            # "pbc" or "obc"
    seed=12345,
    initial_state="neel",      # "neel" or "random"
    # Imbrie: ZZ bonds in [delta_min,delta_max], transverse gamma.
    gamma=1.0, delta_min=0.8, delta_max=1.2,
    # XXZ: Jxy*(SxSx+SySy) + Jz*SzSz.
    jxy=1.0, jz=1.0,
    # For both models: longitudinal field h_i in [-h0,h0].
    h0=3.5,
    tmax=100.0,
    error_control=True, nsteps=None,
    abs_tol=1e-8, rel_tol=1e-8,
    use_gpu=False,
)
print(cfg)

## 2. Prepare and save one run (offline)

This saves the **full-precision** disorder realization and initial spin configuration *before* submission. Running this cell again creates another unique folder; save the printed path. To use an older run later, set `run_dir = Path("runs/<existing-folder>")` instead of preparing a new one.

In [ ]:
run_dir = prepare_run(cfg, root="runs")
config, sample, metadata = load_run(run_dir)
print("Run directory:", run_dir.resolve())
print("Qaptiva bitstring:", sample.bitstring)
print("Z eigenvalues:", sample.spins)
print("Fields h_i:", sample.fields)
if config.model == "imbrie":
    print("Imbrie bond couplings:", sample.deltas)

## 3. Submit AnalogQPU job (optional; requires Qaptiva)

Explicitly set `SUBMIT_TO_QAPTIVA=True` when ready. The returned job ID is recorded in `run.json`. The package avoids resubmitting the same run automatically.

In [ ]:
SUBMIT_TO_QAPTIVA = False   # Set True only on a Qaptiva-enabled environment.
if SUBMIT_TO_QAPTIVA:
    job_id = submit_run(run_dir)
    print("Submitted remote job:", job_id)
else:
    print("Remote submission skipped; saved run is ready for offline ED.")

## 4. Retrieve an existing job (optional; requires server login)

You can restart Jupyter and set `run_dir` to an existing folder, e.g. `Path("runs/imbrie_L8_abcd...")`. The saved job ID, rather than an ID typed into code, is used to retrieve remote observables. QLMaaS handles its own authentication; **do not put passwords in the notebook**.

In [ ]:
RETRIEVE_FROM_QAPTIVA = False
QLMAAS_HOST = "YOUR-QLMAAS-HOST"   # Replace only if retrieval is enabled.
if RETRIEVE_FROM_QAPTIVA:
    times_qpu, C_qpu = retrieve_run(run_dir, host=QLMAAS_HOST)
    print("Retrieved", len(times_qpu), "measurement times;")
    print("C_j(t) shape:", C_qpu.shape)
else:
    print("Remote retrieval skipped.")

## 5. Manual ED: automatically use QPU times when available

ED is completely independent of Qaptiva. If you did not retrieve a remote job, the notebook uses a short, user-configured example time grid instead. For large `L`, dense or Krylov ED can be costly. The default safeguard is `max_L=12`.

In [ ]:
if (Path(run_dir) / "qpu_results.npz").exists():
    times_ed, C_ed = run_ed(run_dir, method="auto", max_L=12, overwrite=True)
else:
    demo_times = np.linspace(0.0, min(load_run(run_dir)[0].tmax, 10.0), 101)
    times_ed, C_ed = run_ed(run_dir, times=demo_times,
                            method="auto", max_L=12, overwrite=True)
print("ED data:", C_ed.shape)
print("C(0) =", C_ed[:, 0].mean())

## 6. Plot site-resolved and averaged dynamics

Plots use local saved files, with no extra remote calls. With QPU data present, ED curves are overlaid automatically. Without QPU data, display offline ED curves.

In [ ]:
if (Path(run_dir) / "qpu_results.npz").exists():
    fig = plot_run(run_dir, kind="sites", sites=[0, 1, 2, 3],
                   save=Path(run_dir) / "sites.png")
    plt.show()
    fig = plot_run(run_dir, kind="average", save=Path(run_dir) / "average.png")
    plt.show()
else:
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    for site in range(min(C_ed.shape[0], 4)):
        axes[0].plot(times_ed, C_ed[site], label=f"Site {site}")
    axes[0].legend()
    axes[0].set(ylabel=r"$C_j(t)$", title="Manual ED: local")
    axes[1].plot(times_ed, C_ed.mean(axis=0))
    axes[1].set(ylabel=r"$C(t)$", title="Manual ED: site average")
    for ax in axes:
        ax.set_xlabel("Time")
        ax.grid(alpha=0.2)
    fig.tight_layout()
    plt.show()

## 7. Difference between ED and AnalogQPU

Compute signed difference `C_j^QPU - C_j^ED`, absolute difference, mean absolute error, RMSE, and maximum absolute error. The comparison **requires** retrieved QPU data; it is intentionally skipped in ED-only mode.

In [ ]:
if (Path(run_dir) / "qpu_results.npz").exists():
    metrics = compare_run(run_dir)
    print("Global max absolute error:", metrics["global_max_absolute_error"])
    print("Global RMSE:", metrics["global_rmse"])
    print("Per-site max absolute errors:", metrics["per_site_max_absolute_error"])
    fig = plot_run(run_dir, kind="error", save=Path(run_dir) / "errors.png")
    plt.show()
    fig = plot_run(run_dir, kind="site_error", sites=[0, 1],
                   save=Path(run_dir) / "site_errors.png")
    plt.show()
else:
    print("ED-only run: comparison will be available after retrieving QPU results.")

---

**Repeatability:** `run.json` stores parameters and the remote job ID; `sample.npz` stores exact input arrays. You can select a different initial state with the same seed in a new run without changing disorder. `qpu_results.npz`, `ed_results.npz`, and `comparison.npz` are local and portable. Refer to the PDF guide for physical definitions and troubleshooting.